# Module 45: PyTorch Profiler Deep Dive

This notebook demonstrates how to use `torch.profiler` to identify
performance bottlenecks in training and inference workloads.

**Topics covered:**
- Basic profiler context manager
- Scheduling warm-up and active phases
- Key averages and sorting
- Chrome trace export
- Memory profiling

In [ ]:
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity, schedule

model = nn.Sequential(
    nn.Linear(1024, 512), nn.ReLU(),
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 10),
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Basic profiling: forward + backward pass
x = torch.randn(64, 1024)
target = torch.randint(0, 10, (64,))
loss_fn = nn.CrossEntropyLoss()

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    output = model(x)
    loss = loss_fn(output, target)
    loss.backward()

print(prof.key_averages().table(sort_by='cpu_time_total', row_limit=10))

In [ ]:
# Profiling with schedule: skip warm-up, capture steady-state
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

with profile(
    activities=[ProfilerActivity.CPU],
    schedule=schedule(wait=1, warmup=2, active=3, repeat=1),
    profile_memory=True,
) as prof:
    for step in range(8):
        x = torch.randn(32, 1024)
        optimizer.zero_grad()
        loss = loss_fn(model(x), torch.randint(0, 10, (32,)))
        loss.backward()
        optimizer.step()
        prof.step()

print(prof.key_averages().table(sort_by='self_cpu_memory_usage', row_limit=10))

In [ ]:
# Group by input shapes to see cost variation
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    for batch_size in [16, 32, 64, 128]:
        _ = model(torch.randn(batch_size, 1024))

print(prof.key_averages(group_by_input_shape=True).table(
    sort_by='cpu_time_total', row_limit=15
))

## Next Steps

- Export traces with `prof.export_chrome_trace('trace.json')` and view in Perfetto UI
- Enable CUDA profiling with `ProfilerActivity.CUDA` on GPU workloads
- Use `tensorboard_trace_handler` for the TensorBoard Profiler plugin
- See `trace_analysis.py` for advanced memory snapshot and CUDA kernel analysis